# Python 基础 ETF 回测 Notebook

这份 Notebook 的目标是把 ETF 日线 CSV 检查、收益率计算、rolling 均线、无未来函数持仓、策略收益、净值曲线和基础绩效指标串成一个最小可运行的回测流程。

> 注意：这是一份学习型回测原型，不是实盘系统。这里暂不处理手续费、滑点、成交限制、停牌、流动性冲击等真实交易细节。

## 0. 策略说明

本 Notebook 使用一个最简单的 `MA20` 均线择时策略：

- 每天收盘后计算 `MA20`。
- 如果 `close > MA20`，认为趋势偏强，下一交易日持有 ETF。
- 如果 `close <= MA20`，认为趋势偏弱，下一交易日空仓。
- 使用 `position = signal.shift(1)`，避免把今天收盘后才知道的信号错误地用于今天已经发生的收益。

核心时间关系如下：

```text
signal[t]   = 第 t 天收盘后才知道的信号
position[t] = 第 t-1 天信号决定的第 t 天持仓
ret[t]      = close[t] / close[t-1] - 1
strategy_ret[t] = position[t] * ret[t]
```

In [3]:
# 1. 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.etf_csv_check import check_etf_csv

DATA_PATH = PROJECT_ROOT / "data" / "sample_etf_daily_long.csv"
DATA_PATH

WindowsPath('D:/Quant/data/sample_etf_daily_long.csv')

## 1. 读取 ETF 日线 CSV

这里先读取示例数据。以后只需要修改 `DATA_PATH`，就可以替换成你自己的 ETF 日线 CSV。

CSV 至少建议包含这些字段：`date`, `open`, `high`, `low`, `close`, `volume`。如果有 `amount`、`symbol` 会更方便后续扩展。

In [4]:
df = pd.read_csv(DATA_PATH)
print("行数:", len(df))
display(df.head())
display(df.tail())

行数: 80


,date,open,high,low,close,volume,amount
0,2024-01-02,0.9972,1.0027,0.9951,0.9966,1094735,1090982
1,2024-01-03,0.9930,0.9987,0.9918,0.9942,1038505,1032436
2,2024-01-04,0.9909,0.9948,0.9780,0.9847,868091,854799
3,2024-01-05,0.9857,0.9932,0.9740,0.9789,1018174,996727
4,2024-01-08,0.9754,0.9979,0.9724,0.9909,879340,871345


,date,open,high,low,close,volume,amount
75,2024-04-16,0.9478,0.9489,0.9328,0.9385,1103077,1035270
76,2024-04-17,0.9383,0.9454,0.9297,0.9313,1250406,1164561
77,2024-04-18,0.9313,0.9377,0.9262,0.9296,1078677,1002786
78,2024-04-19,0.9332,0.9368,0.9269,0.9336,1188698,1109824
79,2024-04-22,0.9329,0.9397,0.9316,0.9365,871400,816072


## 2. 字段检查与类型转换

这一节的目标是确认数据能不能安全进入后续计算。

重点会检查：必要字段是否齐全、日期能否转换、数值列能否转换、日期是否升序、是否存在重复日期，以及关键字段是否有空值。

In [5]:
status = check_etf_csv(DATA_PATH)
assert status == 0, f"CSV 检查未通过，返回码: {status}"

required_cols = ["date", "open", "high", "low", "close", "volume"]
df["date"] = pd.to_datetime(df["date"], errors="coerce")
for col in ["open", "high", "low", "close", "volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
if "amount" in df.columns:
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

sort_cols = ["date"]
if "symbol" in df.columns:
    sort_cols = ["symbol", "date"]
df = df.sort_values(sort_cols).reset_index(drop=True)

print("日期范围:", df["date"].min(), "~", df["date"].max())
print("日期是否升序:", df["date"].is_monotonic_increasing)
print("重复日期数量:", df["date"].duplicated().sum())
print("空值统计:")
# isna()函数的含义是寻找空值，sum()是统计总和
display(df[required_cols].isna().sum())
display(df.dtypes)


1. 文件概览
文件路径: D:\Quant\data\sample_etf_daily_long.csv
读取编码: utf-8-sig
原始行数: 80
原始字段: ['date', 'open', 'high', 'low', 'close', 'volume', 'amount']
标准化字段: ['date', 'open', 'high', 'low', 'close', 'volume', 'amount']

2. 必要字段检查
必要字段完整: ['date', 'open', 'high', 'low', 'close', 'volume']
可选字段存在: ['amount']

3. 类型转换检查
无法解析的日期数量: 0
无法解析为数字的 open 数量: 0
无法解析为数字的 high 数量: 0
无法解析为数字的 low 数量: 0
无法解析为数字的 close 数量: 0
无法解析为数字的 volume 数量: 0
无法解析为数字的 amount 数量: 0

4. 日期检查
日期范围: 2024-01-02 ~ 2024-04-22
是否按日期升序排列: 是
重复日期数量: 0

5. 空值检查
date: 0
open: 0
high: 0
low: 0
close: 0
volume: 0
amount: 0

6. OHLC 价格逻辑检查
价格字段存在空值或无法转数字的行数: 0
价格 <= 0 的行数: 0
OHLC 逻辑异常行数: 0

7. 成交量 / 成交额检查
volume 空值或无法转数字数量: 0
volume <= 0 数量: 0
amount 空值或无法转数字数量: 0
amount <= 0 数量: 0

8. 收益率检查
可计算收益率天数: 79
平均日收益率: -0.076144%
最大单日收益率: 1.231302%
最小单日收益率: -1.179320%

最大收益日:
date=2024-01-16 close=0.9948 ret=1.231302%

最小收益日:
date=2024-04-16 close=0.9385 ret=-1.179320%
绝对日收益率 > 12% 的天数: 0

9. 总结
基础字段体检通过，可以继续做收益率 / 均线 / 回测学习。
日期范围: 2024-01-

date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64

date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
amount             int64
dtype: object

## 3. OHLC 价格逻辑检查

`OHLC` 分别表示开盘价、最高价、最低价和收盘价。正常情况下，它们应该满足下面这些基本关系：

```text
high >= open
high >= close
low <= open
low <= close
high >= low
```

In [6]:
bad_ohlc = df[
    (df["high"] < df["low"]) |
    (df["high"] < df["open"]) |
    (df["high"] < df["close"]) |
    (df["low"] > df["open"]) |
    (df["low"] > df["close"])
]
# axis=1的意思是按row执行
non_positive_price = df[(df[["open", "high", "low", "close"]] <= 0).any(axis=1)]
print("OHLC 逻辑异常行数:", len(bad_ohlc))
print("价格 <= 0 行数:", len(non_positive_price))
display(bad_ohlc.head())

OHLC 逻辑异常行数: 0
价格 <= 0 行数: 0


,date,open,high,low,close,volume,amount


## 4. 计算日收益率 `ret`

这里使用收盘价计算最常见的收盘到收盘日收益率：

```text
ret[t] = close[t] / close[t-1] - 1
```

`pct_change()` 输出的是小数，不是百分号字符串。比如 `0.01` 表示 `1%`。

In [7]:
df["ret"] = df["close"].pct_change()
display(df[["date", "close", "ret"]].head(10))
print("最大日收益率:", f"{df['ret'].max():.2%}")
print("最小日收益率:", f"{df['ret'].min():.2%}")
print("绝对收益率 > 12% 的天数:", (df["ret"].abs() > 0.12).sum())

,date,close,ret
0,2024-01-02,0.9966,NaN
1,2024-01-03,0.9942,-0.002408
2,2024-01-04,0.9847,-0.009555
3,2024-01-05,0.9789,-0.005890
4,2024-01-08,0.9909,0.012259
5,2024-01-09,0.9826,-0.008376
6,2024-01-10,0.9867,0.004173
7,2024-01-11,0.9805,-0.006284
8,2024-01-12,0.9802,-0.000306
9,2024-01-15,0.9827,0.002550


最大日收益率: 1.23%
最小日收益率: -1.18%
绝对收益率 > 12% 的天数: 0


## 5. 计算 rolling 均线

`rolling(20).mean()` 表示取当前行及之前最近 19 行，一共 20 个收盘价做平均。对于日线 ETF，这通常近似表示最近 20 个交易日的平均价格。

In [8]:
df["ma20"] = df["close"].rolling(20).mean()
df["ma60"] = df["close"].rolling(60).mean()
display(df[["date", "close", "ma20", "ma60"]].head(25))

,date,close,ma20,ma60
0,2024-01-02,0.9966,NaN,NaN
1,2024-01-03,0.9942,NaN,NaN
2,2024-01-04,0.9847,NaN,NaN
3,2024-01-05,0.9789,NaN,NaN
4,2024-01-08,0.9909,NaN,NaN
5,2024-01-09,0.9826,NaN,NaN
6,2024-01-10,0.9867,NaN,NaN
7,2024-01-11,0.9805,NaN,NaN
8,2024-01-12,0.9802,NaN,NaN
9,2024-01-15,0.9827,NaN,NaN


## 6. 生成交易信号 `signal`

规则很简单：如果 `close > ma20`，信号为 `True`，表示下一交易日想持有；否则为 `False`，表示下一交易日想空仓。

注意：这里的 `signal` 是用当天收盘价算出来的，所以它只能在当天收盘后才知道。

In [9]:
df["signal"] = df["close"] > df["ma20"]
display(df[["date", "close", "ma20", "signal"]].tail(20))
print("signal=True 天数:", df["signal"].sum())
print("signal=False 天数:", (~df["signal"]).sum())

,date,close,ma20,signal
60,2024-03-26,0.9645,0.965505,False
61,2024-03-27,0.9751,0.965705,True
62,2024-03-28,0.9864,0.965880,True
63,2024-03-29,0.9824,0.966205,True
64,2024-04-01,0.9835,0.966960,True
65,2024-04-02,0.9836,0.967675,True
66,2024-04-03,0.9740,0.967890,True
67,2024-04-04,0.9652,0.968010,False
68,2024-04-05,0.9569,0.968000,False
69,2024-04-08,0.9465,0.967530,False


signal=True 天数: 22
signal=False 天数: 58


## 7. 用 `shift(1)` 构建无未来函数持仓 `position`

如果 `signal[t]` 是第 `t` 天收盘后才知道的，那么它不能直接用于赚第 `t` 天的收益 `ret[t]`。

因此要写成：

```python
position[t] = signal[t-1]
```

含义是：昨天收盘后的信号，决定今天是否持有。

In [10]:
df["position"] = df["signal"].shift(1).fillna(False).astype(bool)
display(df[["date", "close", "ma20", "signal", "position", "ret"]].tail(20))

,date,close,ma20,signal,position,ret
60,2024-03-26,0.9645,0.965505,False,True,-0.008532
61,2024-03-27,0.9751,0.965705,True,False,0.010990
62,2024-03-28,0.9864,0.965880,True,True,0.011589
63,2024-03-29,0.9824,0.966205,True,True,-0.004055
64,2024-04-01,0.9835,0.966960,True,True,0.001120
65,2024-04-02,0.9836,0.967675,True,True,0.000102
66,2024-04-03,0.9740,0.967890,True,True,-0.009760
67,2024-04-04,0.9652,0.968010,False,True,-0.009035
68,2024-04-05,0.9569,0.968000,False,False,-0.008599
69,2024-04-08,0.9465,0.967530,False,False,-0.010868


## 8. 计算策略收益 `strategy_ret`

```text
strategy_ret[t] = position[t] * ret[t]
```

如果今天持有 ETF，策略就获得今天 ETF 的收益；如果今天空仓，策略今天收益记为 `0`。这里暂时不考虑手续费和滑点。

In [11]:
df["strategy_ret"] = (df["position"].astype(int) * df["ret"]).fillna(0)
df["benchmark_ret"] = df["ret"].fillna(0)
display(df[["date", "position", "ret", "strategy_ret", "benchmark_ret"]].head(25))

,date,position,ret,strategy_ret,benchmark_ret
0,2024-01-02,False,NaN,0.0,0.000000
1,2024-01-03,False,-0.002408,-0.0,-0.002408
2,2024-01-04,False,-0.009555,-0.0,-0.009555
3,2024-01-05,False,-0.005890,-0.0,-0.005890
4,2024-01-08,False,0.012259,0.0,0.012259
5,2024-01-09,False,-0.008376,-0.0,-0.008376
6,2024-01-10,False,0.004173,0.0,0.004173
7,2024-01-11,False,-0.006284,-0.0,-0.006284
8,2024-01-12,False,-0.000306,-0.0,-0.000306
9,2024-01-15,False,0.002550,0.0,0.002550


## 9. 生成净值曲线

净值曲线本质上是把每天的收益率连续复利累乘得到的：

```text
equity[t] = equity[t-1] * (1 + strategy_ret[t])
```

## 10. 简单绩效指标

本节会计算几个最常见的绩效指标：总收益率、年化收益率 `CAGR`、年化波动率、最大回撤，以及简化版的 `Sharpe Ratio`（暂不减无风险利率）。

注意：如果样本时间很短，年化指标的参考意义会比较有限，不要过度解读。

In [ ]:
def max_drawdown(equity: pd.Series) -> float:
    # 先计算“到当前为止的历史最高净值”
    running_max = equity.cummax()
    # equity = [1.00, 1.10, 1.05, 1.20, 1.15]
    # running_max = [1.00, 1.10, 1.10, 1.20, 1.20]
    # 回撤 = 当前净值 / 历史最高净值 - 1
    # drawdown 也是一个数组
    drawdown = equity / running_max - 1
    # 最大回撤 = 整段回撤序列里最小的那个值
    return drawdown.min()

def performance_summary(ret: pd.Series, equity: pd.Series, periods_per_year: int = 252) -> dict:
    # 把缺失收益率补成 0，避免第一天 NaN 影响统计
    ret = ret.fillna(0)

    # 总收益率 = 期末净值 / 期初净值 - 1
    total_return = equity.iloc[-1] / equity.iloc[0] - 1

    # 样本期天数，用于后面年化
    n_days = len(ret)

    # 年化收益率 CAGR = 期末净值 ^ (一年交易日数 / 样本天数) - 1
    cagr = equity.iloc[-1] ** (periods_per_year / max(n_days, 1)) - 1

    # 年化波动率 = 日收益率标准差 * sqrt(一年交易日数)
    # .std()：标准差
    # ddof=1：样本标准差写法  意思是分母是n-1, ddof=0的意思是分母是n
    vol = ret.std(ddof=1) * np.sqrt(periods_per_year)

    # Sharpe（简化版，不减无风险利率）
    # = 日均收益率 / 日收益率标准差 * sqrt(一年交易日数)
    sharpe = np.nan if vol == 0 else ret.mean() / ret.std(ddof=1) * np.sqrt(periods_per_year)

    # 最大回撤：调用上面的函数计算
    mdd = max_drawdown(equity)

    return {
        "total_return": total_return,
        "cagr": cagr,
        "annual_vol": vol,
        "max_drawdown": mdd,
        "sharpe_no_rf": sharpe,
    }

# 分别计算策略和买入持有基准的绩效指标，再拼成一张表
summary = pd.DataFrame({
    "strategy": performance_summary(df["strategy_ret"], df["strategy_equity"]),
    "benchmark": performance_summary(df["benchmark_ret"], df["benchmark_equity"]),
}).T

# 下面这些指标用百分比显示更直观
for col in ["total_return", "cagr", "annual_vol", "max_drawdown"]:
    summary[col] = summary[col].map(lambda x: f"{x:.2%}")

# Sharpe 是倍数，不是百分比，保留两位小数即可
summary["sharpe_no_rf"] = summary["sharpe_no_rf"].map(lambda x: f"{x:.2f}")
display(summary)

,total_return,cagr,annual_vol,max_drawdown,sharpe_no_rf
strategy,-1.93%,-5.96%,6.22%,-3.36%,-0.96
benchmark,-6.03%,-17.79%,11.36%,-7.51%,-1.67


## 11. 检查信号、持仓和收益的时间对齐

这一节重点确认下面四个量的时间关系是否对齐：

- `signal`：今天收盘后产生的判断；
- `position`：今天实际持仓，来自昨天的 signal；
- `ret`：今天 ETF 从昨收到今收的收益；
- `strategy_ret`：今天策略收益。

如果 `position[t] == signal[t-1]`，说明这份回测在时点上是合理的，没有直接偷看未来。

In [ ]:
check_cols = ["date", "close", "ma20", "signal", "position", "ret", "strategy_ret"]
display(df[check_cols].iloc[18:30])
expected_position = df["signal"].shift(1).fillna(False).astype(bool)
print("position 是否等于 signal.shift(1):", df["position"].equals(expected_position))

## 12. 本周复盘总结模板

这一节可以留给你自己做学习复盘。

建议围绕下面这些问题来写：

1. 我的数据字段是否完整？
2. 我是否理解 `ret = close.pct_change()`？
3. 我是否理解 `rolling(20).mean()` 的窗口含义？
4. 我是否理解 `signal` 和 `position` 的区别？
5. 我是否理解为什么要 `shift(1)`？
6. 策略和买入持有相比表现如何？
7. 这个初版还有哪些不严谨之处？

当前初版的局限包括：没有手续费、滑点、成交价假设、停牌 / 涨跌停 / 流动性处理，也没有做样本内外划分和参数稳健性测试。另一个要记住的点是：示例数据是人造数据，不代表真实市场。

In [ ]:
review = """
本周我完成了一个单 ETF 的 MA20 择时基础回测 Notebook。
我理解了 CSV 字段检查、日收益率 pct_change、rolling 均线、signal、position、shift(1) 和净值曲线的基本含义。
当前版本只是学习型原型，还没有纳入手续费、滑点、真实成交价、样本外测试和参数稳健性检查。
下一步可以加入最大回撤曲线、手续费模型，以及真实 ETF 数据验证。
"""
print(review)